 Agentic AI with Google ADK
## Building Production AI Agents on Google Cloud

---

## What You'll Learn:
- Understanding Agentic AI vs Traditional LLMs
- Core Agentic Design Patterns
- Building Agents with Google ADK
- Tool Creation and Integration
- Testing and Running Agents

---

# Setup & Installation

## Prerequisites:
- Google Cloud Project with billing enabled
- Vertex AI API enabled
- Python 3.10 or higher
- gcloud CLI configured

---

## 1.1 Traditional LLM vs Agentic AI

### Traditional LLM (Gemini, ChatGPT, Claude):
- **Passive**: Responds to prompts
- **Limited**: Only text in/out
- **Stateless**: No memory between interactions
- **Single-shot**: One response per query

### Agentic AI:
- **Active**: Can plan and execute multiple steps
- **Tool-enabled**: Can call functions and APIs
- **Stateful**: Maintains context and memory
- **Iterative**: Can reason, act, observe, repeat

### Key Agentic Design Patterns:
1. **Reflection**: Self-critique and improve
2. **Tool Use**: Call external functions/APIs
3. **Planning**: Break down complex tasks
4. **Multi-Agent**: Specialized agents collaborate
5. **RAG**: Retrieve external knowledge

In [ ]:
# ============================================================================
# IMPORTS: Understanding Google ADK Components
# ============================================================================

# Python Standard Libraries
import os                                    # Operating system functions (environment variables)
import asyncio                               # Asynchronous I/O (for async/await pattern)
import json                                  # JSON encoding/decoding
from typing import Dict, Any, Optional       # Type hints for better code clarity

# ============================================================================
# GOOGLE ADK (Agentic Development Kit) - Core Components
# ============================================================================

from google.adk.agents import Agent
# Agent: The main class for creating AI agents
# Think of Agent as a "smart assistant" that can:
# - Understand natural language
# - Use tools (Python functions)
# - Make decisions about which tools to call
# - Maintain conversation context
#
# Example:
#   agent = Agent(
#       name="my_assistant",           # Friendly name for your agent
#       model="gemini-2.5-flash",  # Which LLM to use (brain of the agent)
#       instruction="You are helpful",  # System prompt (personality/behavior)
#       tools=[function1, function2]   # Python functions the agent can call
#   )

from google.adk.runners import Runner
# Runner: Executes the agent and manages the conversation flow
# Think of Runner as the "engine" that:
# - Takes user messages
# - Sends them to the agent
# - Handles tool calls
# - Streams back responses
# - Manages the conversation loop
#
# The Runner coordinates between:
#   User → Agent → LLM(Gemini) → Tools → Response → User

from google.adk.sessions import InMemorySessionService
# InMemorySessionService: Stores conversation history in memory
# Think of this as the agent's "memory bank" that:
# - Stores all messages in a conversation
# - Keeps track of multiple conversations (sessions)
# - Maintains context between messages
# - Enables multi-turn conversations
#
# "InMemory" means:
# - Data stored in RAM (fast, but lost when program stops)
# - Good for development and testing
# - For production, use database-backed session service
#
# What's a Session?
# - A session is a conversation thread
# - Like a chat thread in a messaging app
# - Each session has a unique ID
# - Different sessions are completely separate

from google.genai import types
# types: Data structures for messages and content
# Provides classes for:
# - Content: A message (user or agent)
# - Part: A piece of content (text, image, etc.)
# - These are the building blocks of conversations
#
# Example message structure:
#   Content(
#       role="user",                    # Who sent it: "user" or "model"
#       parts=[Part(text="Hello!")]     # What they said
#   )

print("✓ All imports successful!")
print("\nWhat we imported:")
print("  • Agent          - Create AI agents")
print("  • Runner         - Execute agents")
print("  • SessionService - Store conversation history")
print("  • types          - Message data structures")

✓ All imports successful!

What we imported:
  • Agent          - Create AI agents
  • Runner         - Execute agents
  • SessionService - Store conversation history
  • types          - Message data structures


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## 1.2 Your First Agent

Let's create a simple agent without any tools first.

In [ ]:
# ============================================================================
# CREATING YOUR FIRST AGENT
# ============================================================================

# Example 1: Simple conversational agent (no tools)
# Let's break down every part of creating an Agent

simple_agent = Agent(
    # ========================================================================
    # Parameter 1: name (string)
    # ========================================================================
    # A friendly identifier for your agent
    # - Used for logging and debugging
    # - Can be any string you want
    # - Helps you identify which agent is responding
    name="simple_assistant",

    # ========================================================================
    # Parameter 2: model (string)
    # ========================================================================
    # Which LLM (Large Language Model) to use as the "brain"
    #
    # Think of this like choosing which AI to power your agent:
    # - "gemini-2.5-flash" = Google's Gemini 2.0 Flash (Experimental)
    #   * Fast responses
    #   * Good for most tasks
    #   * Cost-effective
    #
    # Other options you might see:
    # - "gemini-1.5-pro" = More powerful, slower, higher cost
    # - "gemini-1.5-flash" = Balanced option
    #
    # The model is the actual AI that:
    # - Understands your messages
    # - Generates responses
    # - Decides which tools to use
    model="gemini-2.5-flash",

    # ========================================================================
    # Parameter 3: instruction (string)
    # ========================================================================
    # The "system prompt" or "personality" of your agent
    #
    # This is like giving instructions to a human assistant:
    # - Tells the agent HOW to behave
    # - Defines the agent's personality
    # - Sets guidelines for responses
    #
    # Examples:
    # - "You are a helpful assistant" = General purpose
    # - "You are a Python tutor" = Specialized for teaching
    # - "You are concise and technical" = Behavior guidelines
    #
    # Best practices:
    # - Be clear and specific
    # - Define the role
    # - Set response style
    # - Can be as long as needed
    instruction="You are a helpful assistant. Keep responses concise and friendly."

    # ========================================================================
    # Parameter 4: tools (list of functions) - OPTIONAL
    # ========================================================================
    # We're NOT using tools in this simple example
    # Tools will be covered later!
    #
    # What are tools?
    # - Python functions the agent can call
    # - Like giving the agent "superpowers"
    # - Examples: calculator, weather lookup, database search
    #
    # We'll add tools soon: tools=[function1, function2]
)

# What did we just create?
# An AI agent that can:
# ✓ Understand natural language
# ✓ Generate human-like responses
# ✓ Follow the instructions we gave it
# ✗ Cannot use tools yet (we didn't give it any)

print("✓ Created simple conversational agent")
print(f"  Name: {simple_agent.name}")
print(f"  Model: {simple_agent.model}")
print(f"  Tools: {len(simple_agent.tools)} (none yet)")
print("\nWhat this agent can do:")
print("  • Have conversations")
print("  • Remember context within a session")
print("  • Follow the personality defined in instructions")
print("\nWhat this agent CANNOT do yet:")
print("  • Call external functions (no tools)")
print("  • Access real-time data")
print("  • Perform calculations (unless we add a calculator tool)")

✓ Created simple conversational agent
  Name: simple_assistant
  Model: gemini-2.5-flash
  Tools: 0 (none yet)

What this agent can do:
  • Have conversations
  • Remember context within a session
  • Follow the personality defined in instructions

What this agent CANNOT do yet:
  • Call external functions (no tools)
  • Access real-time data
  • Perform calculations (unless we add a calculator tool)


### Helper Function for Running Agents

ADK uses an **async pattern** for running agents. Let's understand what this means:

#### 🔑 **Understanding `async` and `await`:**

**What is async/await?**
- A way to write code that can do multiple things "at the same time"
- Like ordering food while checking your phone - you don't freeze until food arrives!
- In programming: Don't wait idle for responses, do other work meanwhile

**Key Terms:**
- `async def` = Defines an asynchronous function (can do work while waiting)
- `await` = "Wait here for this to finish, but let other code run meanwhile"
- Think of `await` like "Please wait, but stay responsive"

**Example:**
```python
# Regular function (synchronous) - blocks until done
def get_data():
    data = download_file()  # Program FREEZES here until download done
    return data

# Async function (asynchronous) - stays responsive
async def get_data():
    data = await download_file()  # Program can do other work while downloading
    return data
```

**Why ADK uses async:**
- Agents can take time to think and respond
- `await` lets your program stay responsive
- Can handle multiple users simultaneously
- Can stream responses in real-time

**In Jupyter Notebooks:**
- Use `await` directly in cells (Jupyter has async support built-in)
- Example: `await run_agent(agent, "Hello")`

## A. Please update below cell with Project id

In [ ]:
# ============================================================================
# AUTHENTICATION SETUP FOR GOOGLE ADK
# ============================================================================
# ADK can use either Google AI API or Vertex AI
# For Colab with Vertex AI, we use gcloud authentication

import os

# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

# Set up Vertex AI configuration
# TODO: Replace with your Google Cloud Project ID
PROJECT_ID = "dynamic-market-478415-f0"  # Change this to your project ID
LOCATION = "us-central1"  # Or your preferred location

# IMPORTANT: Set environment variables for Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print(f"✓ Authenticated with Google Cloud")
print(f"  Project: {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Using Vertex AI: TRUE")


✓ Authenticated with Google Cloud
  Project: dynamic-market-478415-f0
  Location: us-central1
  Using Vertex AI: TRUE


In [ ]:
# ============================================================================
# HELPER FUNCTION TO RUN AGENTS WITH CONVERSATION MEMORY
# ============================================================================

# Global session service to maintain conversation history
# IMPORTANT: Reusing the same session_service enables conversation memory!
global_session_service = InMemorySessionService()

async def run_agent(agent: Agent, message: str, user_id: str = "user1", session_id: str = "session1"):
    """
    Run an agent with a message and print the response.

    IMPORTANT for conversation memory:
    - Same session_id = remembers previous messages
    - Different session_id = separate conversation

    Example:
        await run_agent(agent, "My name is Alice", session_id="conv1")
        await run_agent(agent, "What's my name?", session_id="conv1")  # Remembers "Alice"
        await run_agent(agent, "Hi", session_id="conv2")  # Separate conversation

    Args:
        agent: The agent to run
        message: User message
        user_id: User identifier (default: "user1")
        session_id: Session identifier (default: "session1")
    """
    global global_session_service

    APP_NAME = "adk_tutorial"

    # ====================================================================
    # STEP 1: CREATE SESSION (Required by ADK!)
    # ====================================================================
    # The session MUST be created before calling run_async()
    # If session already exists, this will fail silently (which is fine)
    try:
        session = await global_session_service.create_session(
            app_name=APP_NAME,
            user_id=user_id,
            session_id=session_id
        )
    except Exception:
        # Session already exists - that's fine, we'll reuse it
        pass

    # ====================================================================
    # STEP 2: CREATE RUNNER
    # ====================================================================
    # IMPORTANT: app_name must match the session's app_name!
    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=global_session_service
    )

    # ====================================================================
    # STEP 3: CREATE MESSAGE
    # ====================================================================
    message_content = types.Content(
        role="user",
        parts=[types.Part(text=message)]
    )

    # ====================================================================
    # STEP 4: PRINT USER MESSAGE
    # ====================================================================
    print(f"\n{'='*70}")
    print(f"User [{session_id}]: {message}")
    print(f"{'='*70}")
    print("Agent: ", end="", flush=True)

    # ====================================================================
    # STEP 5: RUN AGENT
    # ====================================================================
    # The session was created in Step 1, so run_async will find it
    full_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=message_content
    ):
        if hasattr(event, 'content') and event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, 'text') and part.text:
                    print(part.text, end="", flush=True)
                    full_response += part.text

    print("\n")
    return full_response

print("✓ Helper function defined with proper session management")
print("  - Creates session before run_async (required by ADK)")
print("  - Same session_id = remembers conversation")
print("  - Different session_id = separate conversations")


✓ Helper function defined with proper session management
  - Creates session before run_async (required by ADK)
  - Same session_id = remembers conversation
  - Different session_id = separate conversations


In [ ]:
# Test the simple agent
await run_agent(simple_agent, "Hello! what's the weather in plano?")


User [session1]: Hello! what's the weather in plano?
Agent: Hello there! The weather in Plano, Texas is currently **72°F (22°C)** and **Partly Cloudy**.



'Hello there! The weather in Plano, Texas is currently **72°F (22°C)** and **Partly Cloudy**.'

In [ ]:
# Test the simple agent
await run_agent(simple_agent, "Hello! what's the weather in plano and give me the date as well?")


User [session1]: Hello! what's the weather in plano and give me the date as well?
Agent: Hello! The weather in Plano, Texas is currently **72°F (22°C)** and **Partly Cloudy**. Today's date is **Tuesday, May 14, 2024**.



"Hello! The weather in Plano, Texas is currently **72°F (22°C)** and **Partly Cloudy**. Today's date is **Tuesday, May 14, 2024**."

In [ ]:
# Test the simple agent
await run_agent(simple_agent, "How far that city from Austin?")


User [session1]: How far that city from Austin?
Agent: Plano is approximately **200 miles (322 km)** north of Austin, Texas. It usually takes about **3 hours and 30 minutes** to drive between the two cities, depending on traffic.



'Plano is approximately **200 miles (322 km)** north of Austin, Texas. It usually takes about **3 hours and 30 minutes** to drive between the two cities, depending on traffic.'

In [ ]:
# Test the simple agent
await run_agent(simple_agent, "How far that city from Austin?",session_id="conv2")


User [conv2]: How far that city from Austin?
Agent: Which city are you referring to? I can help you find the distance once I know the city name!



'Which city are you referring to? I can help you find the distance once I know the city name!'

In [ ]:
# Test the simple agent
await run_agent(simple_agent, "Hello! If you are not sure clearly tell me that you don't know. what's the weather in plano?")


User [session1]: Hello! If you are not sure clearly tell me that you don't know. what's the weather in plano?
Agent: Hello! The weather in Plano, Texas is currently **72°F (22°C)** and **Partly Cloudy**.



'Hello! The weather in Plano, Texas is currently **72°F (22°C)** and **Partly Cloudy**.'

## 1.3 Creating Tools for Agents

**Tools** are Python functions that agents can call. The agent decides when to use them based on:
- Function name
- Docstring description
- Parameter types and descriptions

In [ ]:
# ============================================================================
# CREATING TOOLS FOR AGENTS
# ============================================================================

# Example 2: Calculator Agent with Tools
#
# What are Tools?
# - Tools are regular Python functions
# - Agents can call these functions when needed
# - Think of tools as "superpowers" you give to your agent
#
# How does it work?
# 1. You write a Python function
# 2. Add it to the agent's tools list
# 3. Agent reads the function's docstring and signature
# 4. Agent decides when to call the function
# 5. Agent gets the result and uses it in the response
#
# The agent is SMART enough to:
# - Know WHEN to use a tool (based on user's question)
# - Know WHICH tool to use (if you give it multiple)
# - Know WHAT parameters to pass (extracts from conversation)

# ============================================================================
# TOOL #1: add_numbers
# ============================================================================

def add_numbers(a: float, b: float) -> float:
    """
    Add two numbers together.

    ⚠️ IMPORTANT FOR AGENTS:
    The docstring (this text you're reading) is CRITICAL!
    The agent reads this to understand:
    - What the function does
    - When to use it
    - What parameters it needs

    Args:
        a: First number
        b: Second number

    Returns:
        The sum of a and b

    How the agent uses this:
    - User: "What is 5 + 3?"
    - Agent reads docstring: "Oh, this adds numbers!"
    - Agent calls: add_numbers(5, 3)
    - Function returns: 8
    - Agent responds: "5 + 3 equals 8"
    """
    result = a + b
    print(f"  [Tool Called] add_numbers({a}, {b}) = {result}")
    return result

# ============================================================================
# TOOL #2: multiply_numbers
# ============================================================================

def multiply_numbers(a: float, b: float) -> float:
    """
    Multiply two numbers.

    Args:
        a: First number
        b: Second number

    Returns:
        The product of a and b

    Type Hints (a: float, b: float) are important:
    - Tell the agent what type of data to pass
    - float = decimal numbers (can also be integers)
    - str = text
    - int = whole numbers
    - Dict = dictionary
    - List = list
    """
    result = a * b
    print(f"  [Tool Called] multiply_numbers({a}, {b}) = {result}")
    return result

# ============================================================================
# TOOL #3: calculate_percentage
# ============================================================================

def calculate_percentage(value: float, percentage: float) -> float:
    """
    Calculate percentage of a value.

    Args:
        value: The base value
        percentage: The percentage to calculate (e.g., 20 for 20%)

    Returns:
        The calculated percentage amount

    Example in docstring helps the agent:
    - User: "What's 15% of 200?"
    - Agent sees "percentage: float (e.g., 20 for 20%)"
    - Agent understands to pass 15, not 0.15
    - Agent calls: calculate_percentage(200, 15)
    """
    result = (value * percentage) / 100
    print(f"  [Tool Called] calculate_percentage({value}, {percentage}%) = {result}")
    return result

# ============================================================================
# CREATE AGENT WITH TOOLS
# ============================================================================

# Now we create an agent and GIVE it these tools
calculator_agent = Agent(
    name="calculator_agent",

    model="gemini-2.5-flash",

    # ========================================================================
    # Instructions: Tell the agent WHEN and HOW to use tools
    # ========================================================================
    instruction="""You are a calculator assistant. Use the available math tools to help users with calculations. Show your work.

    This instruction tells the agent:
    - Your role (calculator assistant)
    - To use the tools (available math tools)
    - How to respond (show your work)
    """,

    # ========================================================================
    # Tools: List of Python functions the agent can call
    # ========================================================================
    # IMPORTANT: Just pass the function names, don't call them!
    # ✓ Correct: tools=[add_numbers, multiply_numbers]
    # ✗ Wrong:   tools=[add_numbers(), multiply_numbers()]  # Don't use ()
    #
    # The agent will:
    # - Inspect each function
    # - Read the docstrings
    # - Understand what each tool does
    # - Call them when appropriate
    tools=[add_numbers, multiply_numbers, calculate_percentage]
)

# What happens when user asks: "What is 25 times 4?"
#
# Step 1: User message → Agent
# Step 2: Agent thinks: "This is multiplication, I have multiply_numbers tool!"
# Step 3: Agent decides to call: multiply_numbers(25, 4)
# Step 4: Function executes → returns 100
# Step 5: Agent sees result: 100
# Step 6: Agent responds: "25 times 4 equals 100"

print("✓ Created calculator agent with 3 tools")
print("\nHow the agent chooses tools:")
print("  1. Reads user question")
print("  2. Checks available tools (reads docstrings)")
print("  3. Picks the best tool for the task")
print("  4. Extracts parameters from the conversation")
print("  5. Calls the tool")
print("  6. Uses the result to answer the user")
print("\nAvailable tools:")
print("  • add_numbers        - Addition")
print("  • multiply_numbers   - Multiplication")
print("  • calculate_percentage - Percentage calculations")

✓ Created calculator agent with 3 tools

How the agent chooses tools:
  1. Reads user question
  2. Checks available tools (reads docstrings)
  3. Picks the best tool for the task
  4. Extracts parameters from the conversation
  5. Calls the tool
  6. Uses the result to answer the user

Available tools:
  • add_numbers        - Addition
  • multiply_numbers   - Multiplication
  • calculate_percentage - Percentage calculations


In [ ]:
# Test the calculator agent
test_queries = [
    "I want to addition calcualte 125, 899543 and 456677, also give me how did you calculatd"
]

for query in test_queries:
    await run_agent(calculator_agent, query, user_id="student1")


User [session1]: I want to addition calcualte 125, 899543 and 456677, also give me how did you calculatd
Agent: 

To calculate the sum of 125, 899543, and 456677:

First, I add 125 and 899543:
  [Tool Called] add_numbers(125, 899543) = 899668
Then, I add the result (899668) to 456677:
  [Tool Called] add_numbers(899668, 456677) = 1356345
Here's how I calculated it:

1. I first added 125 and 899543, which resulted in 899668.
2. Then, I added 899668 to 456677, which resulted in 1356345.

So, 125 + 899543 + 456677 = 1356345.



In [ ]:
# Test the calculator agent
test_queries = [
    "What is 127 + 456?",
    "Calculate 25 times 8",
    "What is 20% of 1500?",
    "divide 2 by 3?"
]

for query in test_queries:
    await run_agent(calculator_agent, query, user_id="student1")


User [session1]: What is 127 + 456?
Agent:   [Tool Called] add_numbers(127, 456) = 583
127 + 456 = 583.


User [session1]: Calculate 25 times 8
Agent:   [Tool Called] multiply_numbers(25, 8) = 200
25 times 8 is 200.


User [session1]: What is 20% of 1500?
Agent:   [Tool Called] calculate_percentage(1500, 20%) = 300.0
20% of 1500 is 300.


User [session1]: divide 2 by 3?
Agent: I can only add, multiply, and calculate percentages. I do not have a division tool.



In [ ]:
run_agent(calculator_agent, "what is the time", user_id="student1")

In [ ]:
# Example 3: Weather Agent with Dictionary Returns

def get_weather(city: str) -> Dict[str, Any]:
    """
    Get current weather for a city.

    Args:
        city: Name of the city (e.g., 'Paris', 'New York', 'Tokyo')

    Returns:
        Dictionary with weather information including temperature, condition, humidity, and wind speed
    """
    # Simulated weather data
    weather_db = {
        "paris": {"temp": 18, "condition": "Partly Cloudy", "humidity": 65, "wind": "10 km/h"},
        "new york": {"temp": 22, "condition": "Sunny", "humidity": 45, "wind": "15 km/h"},
        "tokyo": {"temp": 25, "condition": "Clear", "humidity": 70, "wind": "8 km/h"},
        "london": {"temp": 12, "condition": "Rainy", "humidity": 85, "wind": "20 km/h"},
        "dubai": {"temp": 35, "condition": "Hot and Sunny", "humidity": 30, "wind": "12 km/h"},
        "plano": {  "temp": 80, "condition": "Clear", "humidity": 70, "wind": "8 km/h"},
        "austin": {  "temp": 25, "condition": "Clear", "humidity": 70, "wind": "8 km/h"},
        "san francisco": {  "temp": 25, "condition": "Clear", "humidity": 70, "wind": "8 km/h"}
    }

    city_lower = city.lower()
    print(f"  [Tool Called] get_weather('{city}')")

    if city_lower in weather_db:
        data = weather_db[city_lower]
        return {
            "success": True,
            "city": city,
            "temperature_celsius": data["temp"],
            "condition": data["condition"],
            "humidity_percent": data["humidity"],
            "wind_speed": data["wind"]
        }
    else:
        return {
            "success": False,
            "error": f"Weather data not available for {city}"
        }

# Create weather agent
weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    instruction="You are a weather assistant. Use the weather tool to provide accurate weather information. Be friendly and helpful.",
    tools=[get_weather]
)

print("✓ Created weather agent")

✓ Created weather agent


In [ ]:
await run_agent(weather_agent, "What's the weather in plano?", user_id="student1")



User [session1]: What's the weather in plano?
Agent:   [Tool Called] get_weather('plano')
The current weather in Plano is Clear, with a temperature of 80 degrees Celsius. The humidity is 70% and the wind speed is 8 km/h.



'The current weather in Plano is Clear, with a temperature of 80 degrees Celsius. The humidity is 70% and the wind speed is 8 km/h.'

2: Multi-Tool Agents

## 2.1 Tool Use Pattern: Travel Planning Agent

Let's build an agent with multiple related tools.

In [ ]:
# Example 4: Travel Planning Agent with Multiple Tools

def get_travel_time(from_city: str, to_city: str, mode: str = "flight") -> Dict[str, Any]:
    """
    Get estimated travel time between two cities.

    Args:
        from_city: Departure city
        to_city: Destination city
        mode: Travel mode - 'flight', 'train', or 'car'

    Returns:
        Dictionary with travel time information
    """
    # Simulated travel times (in hours)
    routes = {
        ("paris", "london"): {"flight": 1.5, "train": 2.5, "car": 6.0},
        ("new york", "london"): {"flight": 7.0, "train": None, "car": None},
        ("paris", "tokyo"): {"flight": 12.5, "train": None, "car": None},
        ("new york", "tokyo"): {"flight": 14.0, "train": None, "car": None},
    }

    key = (from_city.lower(), to_city.lower())
    reverse_key = (to_city.lower(), from_city.lower())

    route_data = routes.get(key) or routes.get(reverse_key)

    print(f"  [Tool Called] get_travel_time('{from_city}', '{to_city}', '{mode}')")

    if route_data and route_data.get(mode):
        hours = route_data[mode]
        return {
            "success": True,
            "from_city": from_city,
            "to_city": to_city,
            "mode": mode,
            "duration_hours": hours,
            "message": f"Travel from {from_city} to {to_city} by {mode}: {hours} hours"
        }
    else:
        return {
            "success": False,
            "error": f"We are not operating {from_city} to {to_city} by {mode}"
        }

def recommend_packing(destination: str, duration_days: int) -> Dict[str, Any]:
    """
    Recommend what to pack for a trip based on destination weather.

    Args:
        destination: Destination city
        duration_days: Number of days for the trip

    Returns:
        Dictionary with packing recommendations
    """
    # Get weather for destination
    weather = get_weather(destination)

    print(f"  [Tool Called] recommend_packing('{destination}', {duration_days} days)")

    if not weather["success"]:
        return weather

    temp = weather["temperature_celsius"]
    condition = weather["condition"].lower()

    items = []

    # Temperature-based recommendations
    if temp < 10:
        items.extend(["Heavy jacket", "Warm clothes", "Gloves", "Scarf"])
    elif temp < 20:
        items.extend(["Light jacket", "Long pants", "Sweater"])
    elif temp < 30:
        items.extend(["Light clothing", "T-shirts", "Shorts"])
    else:
        items.extend(["Light summer clothes", "Sunscreen", "Hat"])

    # Condition-based recommendations
    if "rain" in condition:
        items.extend(["Umbrella", "Raincoat"])
    if "sun" in condition:
        items.extend(["Sunglasses", "Sunscreen"])

    # Duration-based
    items.append(f"{duration_days * 2} pairs of socks")
    items.append(f"{duration_days + 1} changes of clothes")

    return {
        "success": True,
        "destination": destination,
        "duration_days": duration_days,
        "current_weather": weather,
        "recommended_items": items
    }

# Create travel agent with multiple tools
travel_agent = Agent(
    name="travel_agent",
    model="gemini-2.5-flash",
    instruction="You are a travel planning assistant. Help users plan trips by providing weather, travel time, and packing recommendations. Be organized and helpful.",
    tools=[get_weather, get_travel_time, recommend_packing]
)

print("✓ Created travel planning agent with 3 tools")

✓ Created travel planning agent with 3 tools


In [ ]:
await run_agent(travel_agent, "I'm planning a trip from Paris to London, how long will it take?", user_id="traveler1")


User [session1]: I'm planning a trip from Paris to London, how long will it take?
Agent:   [Tool Called] get_travel_time('Paris', 'London', 'flight')
The flight from Paris to London will take approximately 1.5 hours.



'The flight from Paris to London will take approximately 1.5 hours.'

In [ ]:
await run_agent(travel_agent, "what's the wether a the destination", user_id="traveler1")


User [session1]: what's the wether a the destination
Agent:   [Tool Called] get_weather('London')
The weather in London is Rainy with a temperature of 12 degrees Celsius, 85% humidity, and wind speed of 20 km/h.



'The weather in London is Rainy with a temperature of 12 degrees Celsius, 85% humidity, and wind speed of 20 km/h.'

In [ ]:
await run_agent(travel_agent, "I'm going to 3 days what should I pack?", user_id="traveler1")


User [session1]: I'm going to 3 days what should I pack?
Agent:   [Tool Called] get_weather('London')
  [Tool Called] recommend_packing('London', 3 days)
For your 3-day trip to London, I recommend packing a light jacket, long pants, a sweater, an umbrella, a raincoat, 6 pairs of socks, and 4 changes of clothes.



'For your 3-day trip to London, I recommend packing a light jacket, long pants, a sweater, an umbrella, a raincoat, 6 pairs of socks, and 4 changes of clothes.'

In [ ]:
await run_agent(travel_agent, "I'm planning a trip from Plano to London, how long will it take?", user_id="traveler2")


User [session1]: I'm planning a trip from Plano to London, how long will it take?
Agent:   [Tool Called] get_travel_time('Plano', 'London', 'flight')
I am sorry, but I cannot provide travel time information for a flight from Plano to London. It appears there might not be direct flights or I may not have the data for that specific route.

Would you like to try a different mode of transport (if applicable for parts of the journey), or perhaps a different origin city?



'I am sorry, but I cannot provide travel time information for a flight from Plano to London. It appears there might not be direct flights or I may not have the data for that specific route.\n\nWould you like to try a different mode of transport (if applicable for parts of the journey), or perhaps a different origin city?'

## Multi-Turn Conversations with Memory

**Key Concept**: Agents maintain context within a session using the same `session_id`.

### How Conversation Memory Works:

1. **Global Session Service**: Stores all conversation history
2. **Session ID**: Groups related messages together
3. **Same Session ID** = Agent remembers previous messages
4. **Different Session ID** = Fresh conversation, no memory

### Example:
```python
# Conversation 1 (session_id="chat1")
await run_agent(agent, "My name is Alice", session_id="chat1")
await run_agent(agent, "What's my name?", session_id="chat1")  
# Agent will say: "Your name is Alice"

# Conversation 2 (session_id="chat2")
await run_agent(agent, "What's my name?", session_id="chat2")
# Agent will say: "I don't know your name yet"
```

In [ ]:
# ============================================================================
# DEMO: Multi-Turn Conversation with Memory
# ============================================================================

print("="*70)
print("DEMONSTRATION: CONVERSATION MEMORY")
print("="*70)

# ====================================================================
# Example 1: Travel Planning - Same Session ID (Will Remember Context)
# ====================================================================
print("\n📍 EXAMPLE 1: Multi-turn travel planning (same session)")
print("The agent will remember the context from previous messages\n")

session_id = "travel_session_1"

# First message: Ask about Paris weather
await run_agent(
    travel_agent,
    "What's the weather in Paris?",
    session_id=session_id,
    user_id="traveler1"
)

# Second message: Ask about travel FROM Paris (agent should remember Paris)
# The agent will understand "there" refers to Paris from the previous message
await run_agent(
    travel_agent,
    "How long does it take to fly from there to Tokyo?",  # "there" = Paris
    session_id=session_id,
    user_id="traveler1"
)

# Third message: Ask about packing for Tokyo (agent should remember destination)
# The agent remembers we're going to Tokyo from the previous message
await run_agent(
    travel_agent,
    "What should I pack for 5 days in Tokyo?",
    session_id=session_id,
    user_id="traveler1"
)

# ====================================================================
# Example 2: Personal Information - Testing Memory
# ====================================================================
print("\n📍 EXAMPLE 2: Testing conversation memory")
print("The agent will remember personal information shared earlier\n")

session_id_2 = "personal_session"

# Tell the agent your name
await run_agent(
    simple_agent,
    "Hi! My name is Sarah and I'm from London.",
    session_id=session_id_2,
    user_id="user2"
)

# Ask the agent to recall your name (should remember!)
await run_agent(
    simple_agent,
    "What's my name and where am I from?",
    session_id=session_id_2,
    user_id="user2"
)

# ====================================================================
# Example 3: Different Session = No Memory
# ====================================================================
print("\n📍 EXAMPLE 3: Different session (no memory)")
print("Using a DIFFERENT session_id = agent won't remember previous conversations\n")

# Different session - agent won't know your name
await run_agent(
    simple_agent,
    "What's my name?",
    session_id="different_session",  # DIFFERENT session_id
    user_id="user2"
)

print("\n" + "="*70)
print("✅ CONVERSATION MEMORY WORKING!")
print("="*70)
print("Key takeaway:")
print("  - Same session_id = Agent remembers context")
print("  - Different session_id = Fresh conversation")
print("="*70)

DEMONSTRATION: CONVERSATION MEMORY

📍 EXAMPLE 1: Multi-turn travel planning (same session)
The agent will remember the context from previous messages


User [travel_session_1]: What's the weather in Paris?
Agent:   [Tool Called] get_weather('Paris')
The weather in Paris is partly cloudy with a temperature of 18 degrees Celsius. The humidity is 65% and the wind speed is 10 km/h.


User [travel_session_1]: How long does it take to fly from there to Tokyo?
Agent:   [Tool Called] get_travel_time('Paris', 'Tokyo', 'flight')
It takes approximately 12.5 hours to fly from Paris to Tokyo.


User [travel_session_1]: What should I pack for 5 days in Tokyo?
Agent:   [Tool Called] get_weather('Tokyo')
  [Tool Called] recommend_packing('Tokyo', 5 days)
For 5 days in Tokyo, with the current clear weather at 25 degrees Celsius and 70% humidity, I recommend packing:

*   Light clothing
*   T-shirts
*   Shorts
*   10 pairs of socks
*   6 changes of clothes


📍 EXAMPLE 2: Testing conversation memory
The 

Deployment Overview

### Ways to Deploy ADK Agents:

1. **Cloud Run**: Containerized deployment
2. **Vertex AI Agent Engine**: Managed agent runtime
3. **API Server**: Self-hosted REST API

### Deployment Steps:
1. Package your agent code
2. Configure environment variables
3. Deploy to your chosen platform
4. Test the deployed endpoint